# 04 — Chia bin đơn điệu và đối chiếu với XGBoost

Hai việc. Một, chia bin lại cho bảng WOE đơn điệu theo chiều nghiệp vụ, vì khối 3 đã đo được cái giá của việc đó nằm dưới ngưỡng đo được trong khi bảng điểm đơn điệu là thứ làm mã lý do nói được thành câu. Hai, dựng XGBoost để trả lời câu hỏi đã treo từ khối 3: nếu cây vượt scorecard thì phần vượt đến từ đâu.

Năm dự đoán cho phần một và bốn dự đoán cho phần hai được ghi vào `notes/du_doan_khoi4.md` **trước khi chạy bất cứ phép đo nào** trong notebook này.

Yêu cầu thêm: `pip install xgboost`. Không cần gói `shap`, giá trị SHAP interaction lấy thẳng từ XGBoost.

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, sqlite3

sys.path.insert(0, str(Path.cwd().parent / 'src'))
import config, scorecard, cv_check, monotone_bins, tree_model

pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)

con = sqlite3.connect(scorecard._ro_uri(config.DB_PATH), uri=True)
iv_goc  = pd.read_sql('SELECT * FROM iv_summary', con)
iv_mono = pd.read_sql('SELECT * FROM iv_summary_mono', con)
bmap    = pd.read_sql('SELECT * FROM bin_map', con)
con.close()
print(f'bin_map: {len(bmap)} dong, {bmap.variable.nunique()} bien')
print(monotone_bins.summary(bmap).to_string())
print(f"\ntong bin: {monotone_bins.summary(bmap).bin_goc.sum()} -> {monotone_bins.summary(bmap).bin_mono.sum()}")

bin_map: 70 dong, 9 bien
                   bin_goc  bin_mono  gop
variable                                 
debt_ratio_valid        11         7    4
real_estate_loans        4         2    2
revolving_util          11         9    2
monthly_income          12        10    2
age                     10        10    0
late_60_89               5         5    0
late_30_59               6         6    0
dependents               5         5    0
late_90                  6         6    0

tong bin: 70 -> 60


---
## 1. Chia bin đơn điệu

### Chiều là tham số khai báo, không phải thứ để đoán

Khối 3 đã trả giá cho bài học này: bản đầu của `force_monotone` tự đoán chiều bằng dấu hiệp phương sai có trọng số, đoán sai ở đúng biến đang cần phân xử, và cho ra một kết luận ngược hoàn toàn. Ở bộ này ép sai chiều `revolving_util` làm mất **0,0513 Gini**, gần bằng đúng cái giá của việc bỏ hẳn biến khỏi model (0,0511). Ép đúng chiều thì `d = +0,00006`, tức không đo được chênh lệch nào, và cái giá xấu nhất mà khoảng tin cậy còn cho phép là 0,0013. Chiều quyết định gần như toàn bộ kết quả, còn bản thân phép ép gần như không tốn gì.

Nên chiều nằm trong `config.MONOTONE_DIRECTION`, mỗi dòng kèm một câu nghiệp vụ, và `force_monotone` **bắt buộc** nhận tham số `direction`; truyền tuple thì raise.

WOE ở đây định hướng theo good, nên `"giam"` nghĩa là rủi ro tăng dần theo giá trị biến. Bảy biến khai `"giam"`, riêng `age` và `monthly_income` khai `"tang"` vì tuổi cao và thu nhập cao là an toàn hơn.

### Phân công giữa Python và SQL

Python quyết định **gộp bin nào**: pool-adjacent-violators là thuật toán vòng lặp, viết bằng SQL không cho ra hơn. SQL **tính lại WOE** trên các bin đã gộp, vẫn bằng một `GROUP BY` và một `CROSS JOIN` với dòng tổng y hệt khối 2, nên không con số WOE nào ra đời ngoài SQL.

Cầu nối là bảng `bin_map`. Nhờ vậy cách gộp bin là một artefact nhìn được và kiểm được thay vì nằm ẩn trong một hàm. Nhãn bin gộp giữ dạng `01+02+03` chứ không đánh số lại từ 1, để bảng điểm và mã lý do vẫn truy ngược được về bin gốc.

Một chi tiết dễ sai: **WOE của bin gộp không bằng trung bình WOE các bin thành phần.** Nó được tính lại từ số good/bad cộng dồn. Giá trị trung bình có trọng số mà PAVA trả về chỉ dùng để xác định nhóm nào gộp với nhóm nào, không phải để làm WOE.

In [2]:
# PAVA gop nhung bin nao, tren train
bins_m, woe_m, wide_m = scorecard.load_data(mono=True)
bins_g, woe_g, _      = scorecard.load_data(mono=False)
for v in sorted(config.MONOTONE_DIRECTION):
    g = bmap[bmap.variable == v]
    # nhom da gop = nhan bin_mono nao duoc anh xa tu HON MOT bin goc. Khong duoc
    # nhan dien bang "co dau +" trong nhan: '3+' va '5+' la nhan bin GOC von da
    # chua dau cong, va bat theo dau cong se bao nham la chung bi gop.
    dem = g.groupby('bin_mono').bin.size()
    gop = sorted(dem[dem > 1].index, key=cv_check.bin_order)
    print(f"{v:20s} {config.MONOTONE_DIRECTION[v]:5s} "
          f"{g.bin.nunique():2d} -> {g.bin_mono.nunique():2d} bin   {'  '.join(gop) or '(khong gop gi)'}")

age                  tang  10 -> 10 bin   (khong gop gi)
debt_ratio_valid     giam  11 ->  7 bin   02+03+04+05+06
dependents           giam   5 ->  5 bin   (khong gop gi)
late_30_59           giam   6 ->  6 bin   (khong gop gi)
late_60_89           giam   5 ->  5 bin   (khong gop gi)
late_90              giam   6 ->  6 bin   (khong gop gi)
monthly_income       tang  12 -> 10 bin   01+02  09+10
real_estate_loans    giam   4 ->  2 bin   0+1+2
revolving_util       giam  11 ->  9 bin   01+02+03


In [3]:
# Chi phi cua viec ep don dieu ca 9 bien cung luc, 5-fold CV trong train
V = sorted(woe_g.variable.unique())
base = cv_check.gini_cv(bins_g, V)
g    = cv_check.gini_cv(bins_g, V, monotone=config.MONOTONE_DIRECTION)
r    = cv_check.paired(base, g)
print(f'bin goc     : Gini CV = {base.mean():.5f}')
print(f'don dieu    : Gini CV = {g.mean():.5f}')
print(f'   d = {r["d"]:+.5f}  se = {r["se"]:.5f}  t = {r["t"]:+.2f}  '
      f'KTC95 = [{r["ktc_lo"]:+.5f}; {r["ktc_hi"]:+.5f}]')
print(f'   cai gia xau nhat (-ktc_lo) = {-r["ktc_lo"]:.5f}')

bin goc     : Gini CV = 0.71518
don dieu    : Gini CV = 0.71500
   d = -0.00019  se = 0.00060  t = -0.31  KTC95 = [-0.00186; +0.00148]
   cai gia xau nhat (-ktc_lo) = 0.00186


Cái giá xấu nhất mà dữ liệu còn cho phép là **0,0019 Gini**, và nhắc lại cách đọc vì rất dễ lấy nhầm đầu: `d = Gini(phương án) − Gini(model đủ)` nên cái giá là `−d`, và cái giá xấu nhất là **đầu trái** của khoảng tin cậy. Đầu phải là mức được lợi tối đa.

Trong notebook này có **hai** con số cho model đơn điệu và chúng khác nhau ở chữ số thứ năm, nên nói rõ để không ai tưởng là lệch: **0,71500** ở trên là chạy PAVA lại **trong từng fold**, tức đo cái giá của cả *quy trình*; **0,71507** dùng ở mục 3 là đọc thẳng bảng `bin_map` đã đóng băng từ toàn bộ train, tức đo cái giá của *bộ bin cụ thể sẽ đem đi triển khai*. Con số thứ nhất là con số trung thực để quyết định có nên đơn điệu hoá không; con số thứ hai là mốc đúng để so với XGBoost, vì XGBoost cũng nhận đúng bộ bin đóng băng đó. Hai cách chênh nhau 7e-05.

Con số 0,0019 nhỏ hơn một bậc so với 0,0063 đến 0,0205 mà khối 2 đo đơn biến, và nhỏ hơn nhiều so với ±0,028 là khoảng tin cậy của chính Gini trên tập OOT. Ở quy mô dữ liệu này nó không đo được.

Ba dự đoán ghi trước cho phần này:

| # | dự đoán | kết quả |
|---|---|---|
| A1 | `d` trong [−0,003; +0,002], cái giá dưới 0,004 | **trúng**, d = −0,00019, cái giá 0,0019 |
| A2 | sáu biến `age`, `monthly_income`, `dependents` và ba biến `late_*` không gộp bin nào hoặc chỉ gộp một cặp | **trúng một phần**: năm biến đầu không gộp gì, nhưng `monthly_income` gộp **hai** cặp |
| A3 | số bin còn 60 đến 64 | **trúng** ở biên dưới: 70 → 60. Ba biến tôi nêu đích danh đều đúng số bin mất (`debt_ratio_valid` −4, `revolving_util` −2, `real_estate_loans` −2); tôi bỏ sót `monthly_income` −2 |

In [4]:
# So sanh truoc/sau tren cung mot khuon
def dung_scorecard(bins, woe):
    C = sorted(woe.variable.unique()); W = scorecard.woe_matrix(bins, woe)
    tr, te = bins.split == 'train', bins.split == 'test'; y = 1 - bins.target
    lr, coef, se = scorecard.fit_logit(W[tr], y[tr])
    f, o = scorecard.scaling_constants()
    pts = scorecard.build_points(coef, C, woe, f, o)
    sc  = scorecard.apply_points(bins, pts)
    return dict(C=C, W=W, coef=coef, se=se, pts=pts, lr=lr, y=y, tr=tr, te=te,
                b=bins.assign(score=sc))

S = {'goc': dung_scorecard(bins_g, woe_g), 'mono': dung_scorecard(bins_m, woe_m)}
rows = []
for t, s in S.items():
    p = {k: s['lr'].predict_proba(s['W'][s[k]])[:, 1] for k in ('tr', 'te')}
    rej = s['b'][s['te'] & (s['b'].score < 580)]
    ly1 = pd.Series([x[0][0] for x in scorecard.reason_codes(rej, s['pts'], top_k=1) if x])
    rows.append({'model': t, 'so bin': len(s['pts']),
                 'train Gini': scorecard.gini(s['y'][s['tr']], p['tr']),
                 'test Gini':  scorecard.gini(s['y'][s['te']], p['te']),
                 'test KS':    scorecard.ks(s['y'][s['te']], p['te']),
                 'tong diem':  f"{s['pts'].groupby('variable').diem.min().sum()}-"
                               f"{s['pts'].groupby('variable').diem.max().sum()}",
                 'ly do so 1 tap trung %': round(ly1.value_counts().iloc[0] / len(rej) * 100, 1)})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print()
bd = pd.DataFrame({t: S[t]['pts'].groupby('variable').diem.agg(lambda s: s.max() - s.min()) for t in S})
print(bd.assign(doi=bd['mono'] - bd['goc']).sort_values('goc', ascending=False).to_string())

model  so bin  train Gini  test Gini  test KS tong diem  ly do so 1 tap trung %
  goc      70      0.7165     0.6984   0.5474   385-640                    84.7
 mono      60      0.7162     0.7006   0.5440   376-636                    74.3

                   goc  mono  doi
variable                         
revolving_util      57    53   -4
late_90             52    54    2
late_30_59          52    51   -1
late_60_89          36    36    0
debt_ratio_valid    21    15   -6
age                 19    22    3
real_estate_loans    9    15    6
dependents           6     4   -2
monthly_income       3    10    7


Hai dự đoán còn lại của phần A đều **trượt, và trượt sát về phía tốt hơn tôi tưởng**.

| # | dự đoán | kết quả |
|---|---|---|
| A4 | lý do số 1 vẫn tập trung trên 75% | **trượt**: 84,7% → **74,3%** |
| A5 | Gini test trong [0,695; 0,700] | **trượt**: **0,7006** |

Về A5 thì phải nói cho chặt: **0,7006 so với 0,6984 không được đọc là cải thiện.** CV nói `d` không phân biệt được với 0, và chênh lệch 0,0022 nằm sâu trong khoảng tin cậy của chính Gini trên tập test (±0,025 theo Hanley–McNeil, ±0,021 theo bootstrap). Kết luận đúng là không đo được khác biệt. Cộng với lần đoán Gini bi quan ở khối 3, đây là lần thứ ba liên tiếp tôi ước lượng thấp, và đó là thông tin về cách tôi đoán chứ không phải về dữ liệu.

Biên độ điểm dịch chuyển đúng chiều mong đợi: `revolving_util` co từ 57 xuống 53 vì mất hai bin ở đầu an toàn, còn `monthly_income` nở từ 3 lên 10 và `real_estate_loans` từ 9 lên 15. Hai biến sau nở ra là chuyện cần giải thích, và mục 2 làm việc đó.

---
## 2. Nghịch lý IV: một biến "vô dụng" lại là biến quan trọng thứ sáu

Sau khi gộp, `real_estate_loans` chỉ còn 2 bin và IV rơi từ 0,0564 xuống **0,0052**, tức dưới ngưỡng 0,02 mà mọi giáo trình dùng để loại biến. Một quy trình chọn biến bằng bảng IV sẽ vứt nó đi.

Đo bằng CV trên chính bộ bin đó thì nó là biến quan trọng thứ sáu trên chín.

In [5]:
iv = iv_goc.set_index('variable')[['n_bins','iv']].join(
     iv_mono.set_index('variable')[['n_bins','iv']], lsuffix='_goc', rsuffix='_mono', how='right')
print(iv.assign(doi_iv=(iv.iv_mono - iv.iv_goc).round(4)).sort_values('iv_mono', ascending=False).to_string())

Vm = sorted(woe_m.variable.unique())
base_m = cv_check.gini_cv(bins_m, Vm)
print(f'\nmodel don dieu: Gini CV = {base_m.mean():.5f}\n')
loo = []
for v in Vm:
    gg = cv_check.gini_cv(bins_m, [x for x in Vm if x != v])
    loo.append(dict(bien=v, iv=float(iv.loc[v, 'iv_mono']), khi_bo=gg.mean(),
                    **cv_check.paired(base_m, gg)))
print(pd.DataFrame(loo).sort_values('d').round(5).to_string(index=False))

                   n_bins_goc  iv_goc  n_bins_mono  iv_mono  doi_iv
variable                                                           
revolving_util             11  1.1354            9   1.1146 -0.0208
late_90                     6  0.8697            6   0.8697  0.0000
late_30_59                  6  0.7687            6   0.7687  0.0000
late_60_89                  5  0.5980            5   0.5980  0.0000
age                        10  0.2545           10   0.2545  0.0000
monthly_income             12  0.0748           10   0.0745 -0.0003
debt_ratio_valid           11  0.0781            7   0.0723 -0.0058
dependents                  5  0.0364            5   0.0364  0.0000
real_estate_loans           4  0.0564            2   0.0052 -0.0512

model don dieu: Gini CV = 0.71507

             bien     iv  khi_bo        d      se         t   ktc_lo   ktc_hi  cung_dau
   revolving_util 1.1146 0.65330 -0.06177 0.00091 -67.53911 -0.06431 -0.05923      True
       late_30_59 0.7687 0.68342 -0.0316

`real_estate_loans` có IV **0,0052** mà bỏ nó ra khỏi model mất **0,0029 Gini** (t = −4,58, năm fold cùng dấu), xếp trên `debt_ratio_valid` (IV gấp 14 lần) và `monthly_income` (IV gấp 14 lần).

Hệ số của nó cũng nhảy từ 0,55 lên **1,88**, vượt xa kỳ vọng "quanh 1" mà cả dự án dùng làm quy tắc kiểm. Đó là một con số lạ nên phải đi kiểm bằng số chứ không kể chuyện.

In [6]:
# He so cua real_estate_loans khi them dan tung bien khac vao model
W, y, tr = S['mono']['W'], S['mono']['y'], S['mono']['tr']
V0 = 'real_estate_loans'
for s in [[V0], [V0,'monthly_income'], [V0,'age'], [V0,'monthly_income','age'], list(W.columns)]:
    _, cf, se = scorecard.fit_logit(W[tr][s], y[tr]); i = s.index(V0) + 1
    print(f"   {('day du 9 bien' if len(s)==9 else ' + '.join(s)):44s} he so = {cf[i]:6.3f}   z = {cf[i]/se[i]:5.2f}")
print('\nTuong quan WOE cua real_estate_loans voi cac bien khac (train):')
print('   ' + '  '.join(f'{k}={v:+.3f}' for k, v in W[tr].corr()[V0].drop(V0).sort_values().items()))

# Co che dang sau suppression: nhom nhieu bat dong san giau hon han
ap0 = pd.read_sql('SELECT split, real_estate_loans, monthly_income, target FROM applications',
                  sqlite3.connect(scorecard._ro_uri(config.DB_PATH), uri=True))
t0 = ap0[ap0.split == 'train']; nhieu = t0.real_estate_loans >= 3
print(f"\nThu nhap trung vi tren train: nhom >=3 khoan = {t0.loc[nhieu,'monthly_income'].median():,.0f}"
      f"   phan con lai = {t0.loc[~nhieu,'monthly_income'].median():,.0f}")

   real_estate_loans                            he so =  1.000   z =  6.14
   real_estate_loans + monthly_income           he so =  1.758   z = 10.54
   real_estate_loans + age                      he so =  1.166   z =  7.10
   real_estate_loans + monthly_income + age     he so =  1.806   z = 10.75
   day du 9 bien                                he so =  1.877   z =  9.66

Tuong quan WOE cua real_estate_loans voi cac bien khac (train):
   monthly_income=-0.165  late_90=-0.030  revolving_util=-0.018  late_60_89=-0.010  age=-0.001  late_30_59=+0.028  dependents=+0.058  debt_ratio_valid=+0.219

Thu nhap trung vi tren train: nhom >=3 khoan = 9,032   phan con lai = 5,166


Một mình thì hệ số đúng bằng **1,000**, tức quy tắc "quanh 1" không hề sai, và nó đúng tới ba chữ số. Con số 1,88 là **suppression**, và gần như toàn bộ mức nhảy đến từ việc thêm `monthly_income` (1,000 → 1,758), trong khi thêm `age` chỉ đưa lên 1,166.

Cơ chế kiểm được: nhóm có từ 3 khoản bất động sản trở lên có thu nhập trung vị **9.032** so với **5.166** của phần còn lại, và tương quan WOE giữa hai biến là **−0,165**. Thu nhập cao che mất rủi ro đòn bẩy, nên đo đơn biến thì hai hiệu ứng triệt tiêu nhau một phần; đưa thu nhập vào model rồi thì hiệu ứng đòn bẩy hiện ra gần gấp đôi.

**Và đây chính là lý do IV không nhìn thấy: IV là đại lượng đơn biến.** Nó không có cơ chế nào để thấy một hiệu ứng bị biến khác che.

Bảng đóng góp biên xáo trộn mạnh sau khi đổi bin, và chỗ này tôi suýt nói sai. Ba thay đổi lớn nhất: `revolving_util` từ −0,05107 xuống −0,06177, `debt_ratio_valid` từ −0,00631 lên −0,00218, `monthly_income` từ −0,00016 xuống −0,00134. **Không được cộng chúng lại** rồi đối chiếu với `d` tổng −0,00019: đóng góp leave-one-out không cộng được, đúng như chính khối 3 chứng minh bằng số khi tổng Gini đơn biến của mười biến ra 2,41. Cộng thử thì ra khoảng −0,0105, tức lệch hơn năm chục lần so với `d` tổng −0,00019, và đó là bằng chứng cho tính không cộng được chứ không phải cho một sai sót ở đâu đó.

Điều đọc được là: **toàn bộ bảng đóng góp biên xáo trộn mạnh trong khi Gini tổng đứng yên.** Bản thân điều đó là một kết quả: "biến này quan trọng thứ mấy" không phải thuộc tính của biến, nó là thuộc tính của cặp (biến, cách chia bin) đặt trong một model cụ thể, y hệt như IV.

Một biến mới rơi vào tình trạng đóng góp bằng không: `dependents`, `d` = +0,00004. Đúng chỗ `open_credit_lines` đứng trước khi bị loại ở khối 3, khác một điểm là hệ số của nó vẫn dương (0,1794, z = 2,38), không lật dấu. Giữ lại theo đúng lý lẽ đã dùng ở khối 3, ghi lại để ai đọc biết là đã cân nhắc.

In [7]:
# He so 0,55 -> 1,88 doi CA hai thu: bo bin va so bien trong model. Tach bang 2x2.
# Ca bon o deu tinh lai o day, khong o nao hardcode: ban truoc giu san 1,019 va
# 0,547 tu mot lan chay cu, va den khi sua loi hoi tu o fit_logit thi hai con so
# do thanh sai ma khong co gi bao.
V0 = 'real_estate_loans'
fw_cu = pd.read_sql('SELECT * FROM features_woe',
                    sqlite3.connect(scorecard._ro_uri(config.DB_PATH), uri=True)
                    ).sort_values('id').reset_index(drop=True)
tcu = fw_cu.split == 'train'; ycu = (1 - fw_cu.target)[tcu]
cot9 = [c for c in fw_cu.columns if c.startswith('woe_') and c != 'woe_open_credit_lines']
_, a1, _ = scorecard.fit_logit(fw_cu.loc[tcu, ['woe_' + V0]], ycu)
_, a9, _ = scorecard.fit_logit(fw_cu.loc[tcu, cot9], ycu)
Wm = scorecard.woe_matrix(bins_m, woe_m); tm = bins_m.split == 'train'; ym = (1 - bins_m.target)[tm]
_, b1, _ = scorecard.fit_logit(Wm.loc[tm, [V0]], ym)
_, b9, _ = scorecard.fit_logit(Wm[tm], ym)
i, j = cot9.index('woe_' + V0) + 1, sorted(woe_m.variable.unique()).index(V0) + 1
print(f"{'':12s}{'bin cu (4 bin)':>18s}{'bin moi (2 bin)':>18s}")
print(f"{'don bien':12s}{a1[1]:18.3f}{b1[1]:18.3f}")
print(f"{'day du':12s}{a9[i]:18.3f}{b9[j]:18.3f}")


                bin cu (4 bin)   bin moi (2 bin)
don bien                 1.000             1.000
day du                   0.548             1.877


Bảng 2×2 tách được hai nguyên nhân, và nó cần thiết vì câu "hệ số nhảy từ 0,55 lên 1,88" so hai con số ở **hai bộ bin khác nhau**, tức không phải suppression thuần:

| | bin cũ (4 bin) | bin mới (2 bin) |
|---|---|---|
| **đơn biến** | 1,000 | 1,000 |
| **đầy đủ 9 biến** | 0,548 | 1,877 |

Hàng trên bằng nhau **chính xác**: đơn biến thì hệ số đúng bằng 1,000 ở **cả hai** bộ bin, đúng như lý thuyết WOE nói. Gộp bin không tự nó làm hệ số to lên. (Bản đầu của bảng này ghi 1,003 và 1,019, và hai con số đó là hệ quả của lỗi hội tụ trong `fit_logit`; sau khi sửa thì hàng trên khít lại đúng như lý thuyết đòi hỏi.)

Hàng dưới là chỗ có nội dung, và **dấu của hiệu chỉnh đa biến đảo ngược theo cách chia bin**. Trên bin cũ, đưa 8 biến kia vào làm hệ số **co lại** 1,000 → 0,548, vì cột WOE chữ U trùng lặp với biến khác (khối 3 đã chứng minh nhánh trái chỉ là bóng của `revolving_util`). Trên bin mới, đưa đúng 8 biến đó vào làm hệ số **nở ra** 1,000 → 1,877, vì contrast còn lại bị `monthly_income` che.

Câu đúng vì thế không phải "gộp bin làm hệ số nhảy lên", mà là: **gộp bin bỏ đi phần trùng lặp, và phần còn lại hoá ra là phần bị che.**

---
## 3. XGBoost: phần vượt đến từ đâu

Ở khối 3 tôi ghi trước rằng nếu XGBoost vượt scorecard thì phần vượt chỉ có thể đến từ ba chỗ. Cách kiểm là dựng ba model cây, **mỗi bước chỉ đổi đúng một thứ**, nên phần tăng của từng bước gán được cho đúng một nguyên nhân:

| model | đầu vào | bước này thêm gì |
|---|---|---|
| scorecard | 9 cột WOE đơn điệu | mốc so |
| M2 | **cùng 9 cột WOE đó** | cây thay cho hàm cộng tính, tức **tương tác** |
| M3 | 9 biến gốc + cờ | cây tự chọn điểm cắt, tức **chỗ cắt bin** |
| M4 | thêm `open_credit_lines`, `debt_ratio` thô | **biến scorecard không dùng** |

So sánh chính là **scorecard với M3**, vì hai bên dùng cùng một bộ 9 biến. M4 để riêng như một câu hỏi khác, không trộn vào.

Ba model cây dùng **đúng bộ fold** của `cv_check.make_folds`, nhờ vậy so theo cặp còn nghĩa: phương sai do fold (biên độ 0,021 Gini ở bộ này) triệt tiêu khi trừ theo từng fold.

**Một thiên lệch cố ý.** Siêu tham số của cây được chọn bằng chính bộ 5-fold đó, nên con số CV của cây lạc quan hơn thực tế, còn scorecard không được ưu ái gì. Chọn hướng thiên lệch ngược với kết luận mình mong đợi là cách rẻ nhất để một phép so sánh tự bảo vệ được: nếu với thiên lệch nghiêng về phía cây mà cây vẫn không vượt được bao nhiêu, kết luận càng chắc.

In [8]:
con_ = sqlite3.connect(scorecard._ro_uri(config.DB_PATH), uri=True)
ap_ = pd.read_sql('SELECT * FROM applications', con_).sort_values('id').reset_index(drop=True)
fw_ = pd.read_sql('SELECT * FROM features_woe_mono', con_).sort_values('id').reset_index(drop=True)
con_.close()
Xs, split, yv = tree_model.feature_sets(ap_, fw_)
trv = split == 'train'

# make_folds chia theo VI TRI dong, khong theo id. Nen moi phep so sanh theo cap
# giua scorecard va cay o duoi chi dung neu dong thu i cua hai ben la cung mot ho
# so. Dieu do khong tu dung: bins_m di theo thu tu bang applications tra ve, con
# Xs da duoc sort theo id. Kiem thang o day thay vi tin.
assert (bins_m.index.values == ap_.id.values).all(), 'thu tu dong scorecard != cay'
assert (bins_m.split.values == split).all(), 'cot split khong khop'
print(f'thu tu dong khop: {len(ap_)} ho so, train {trv.sum()}')

chon = {}
for name in ['M2_woe', 'M3_goc', 'M4_all']:
    print(f'=== {name} ===')
    prm, gg = tree_model.search(Xs[name][trv], yv[trv])
    chon[name] = (prm, gg)
    print(f'   -> {gg.mean():.5f}\n')


thu tu dong khop: 149999 ho so, train 104999
=== M2_woe ===
   {'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 250, 'min_child_weight': 50, 'subsample': 0.8, 'colsample_bytree': 0.8}  ->  0.72492
   {'max_depth': 4, 'learning_rate': 0.1, 'n_estimators': 250, 'min_child_weight': 50, 'subsample': 0.8, 'colsample_bytree': 0.8}  ->  0.72466
   {'max_depth': 5, 'learning_rate': 0.05, 'n_estimators': 500, 'min_child_weight': 50, 'subsample': 0.8, 'colsample_bytree': 0.8}  ->  0.72344
   {'max_depth': 6, 'learning_rate': 0.05, 'n_estimators': 500, 'min_child_weight': 20, 'subsample': 0.8, 'colsample_bytree': 0.8}  ->  0.71963
   {'max_depth': 4, 'learning_rate': 0.05, 'n_estimators': 500, 'min_child_weight': 200, 'subsample': 0.8, 'colsample_bytree': 0.8}  ->  0.72345
   -> 0.72492

=== M3_goc ===
   {'max_depth': 3, 'learning_rate': 0.1, 'n_estimators': 250, 'min_child_weight': 50, 'subsample': 0.8, 'colsample_bytree': 0.8}  ->  0.72935
   {'max_depth': 4, 'learning_rate': 0.1, 'n_est

In [9]:
mc = base_m                      # scorecard don dieu, cung bo fold
bang = [('scorecard (logistic, WOE don dieu)', mc, None)]
for name, nhan in [('M2_woe', 'M2 XGB tren cung cot WOE'),
                   ('M3_goc', 'M3 XGB tren bien goc'),
                   ('M4_all', 'M4 XGB tren tat ca')]:
    bang.append((nhan, chon[name][1], None))
out = []
for i, (nhan, g, _) in enumerate(bang):
    r = cv_check.paired(bang[i-1][1], g) if i else None
    out.append({'model': nhan, 'Gini CV': g.mean(),
                'tang so voi dong tren': r['d'] if r else np.nan,
                't': r['t'] if r else np.nan})
print(pd.DataFrame(out).round(5).to_string(index=False))
r13 = cv_check.paired(mc, chon['M3_goc'][1])
print(f"\nSO SANH CHINH, cung 9 bien: scorecard {mc.mean():.5f} -> M3 {chon['M3_goc'][1].mean():.5f}"
      f"   d = {r13['d']:+.5f}  t = {r13['t']:.2f}  KTC[{r13['ktc_lo']:+.5f}; {r13['ktc_hi']:+.5f}]")

                             model  Gini CV  tang so voi dong tren        t
scorecard (logistic, WOE don dieu)  0.71507                    NaN      NaN
          M2 XGB tren cung cot WOE  0.72492                0.00985 13.03082
              M3 XGB tren bien goc  0.72935                0.00442 10.01792
                M4 XGB tren tat ca  0.73280                0.00346  8.26061

SO SANH CHINH, cung 9 bien: scorecard 0.71507 -> M3 0.72935   d = +0.01428  t = 19.20  KTC[+0.01221; +0.01634]


Phần hơn tách làm hai: **+0,0099 ở bước M2 so với scorecard** (hai bên dùng y hệt một ma trận đầu vào) và phần còn lại từ chỗ cắt bin. Tôi định gọi con số đầu là "tương tác", nhưng bước đó đổi hai thứ cùng lúc chứ không phải một, nên mục ngay dưới kiểm chuyện đó trước đã.

Cả ba model đều thắng ở **cùng một cấu hình** (`max_depth=3`, `lr=0,10`, 250 vòng), nên ba bước trong bảng chỉ đổi đầu vào chứ không lẫn "đổi cấu hình". Đó không phải may: xem mục về early stopping ở dưới.

In [10]:
# Do do on dinh cua phep phan ra, hai nguon nhieu
Xtr9 = Xs['M3_goc'][trv].reset_index(drop=True); ytr9 = yv[trv]
P_ = dict(max_depth=4, learning_rate=0.10, n_estimators=250, min_child_weight=50, subsample=0.8, colsample_bytree=0.8)
print('1. Doi random_state cua XGBoost, giu nguyen bo fold va sieu tham so:')
vals = []
for s in [42, 1, 7, 2024, 99]:
    g_ = tree_model.cv_gini(Xtr9, ytr9, P_, seed_model=s)
    vals.append(g_.mean()); print(f'   random_state={s:<5d} M3 Gini CV = {g_.mean():.5f}')
v = np.array(vals)
print(f'   do lech chuan {v.std(ddof=1):.5f}   bien do {v.max()-v.min():.5f}')

print('\n2. Co dinh MOT bo sieu tham so cho ca ba model, chi con dau vao thay doi:')
Z4_ = pd.concat([Xs['M3_goc'], ap_[['open_credit_lines', 'debt_ratio']]], axis=1)
for P2, nhan in [(P_, 'depth4 lr0.10'),
                 (dict(max_depth=5, learning_rate=0.05, n_estimators=500, min_child_weight=50,
                       subsample=0.8, colsample_bytree=0.8), 'depth5 lr0.05')]:
    r2 = {k: tree_model.cv_gini(Xs[k][trv], yv[trv], P2) for k in ['M2_woe', 'M3_goc']}
    r2['M4_all'] = tree_model.cv_gini(Z4_[trv], yv[trv], P2)
    a = cv_check.paired(r2['M2_woe'], r2['M3_goc']); b = cv_check.paired(r2['M3_goc'], r2['M4_all'])
    print(f"   [{nhan}] M2={r2['M2_woe'].mean():.5f} M3={r2['M3_goc'].mean():.5f} M4={r2['M4_all'].mean():.5f}"
          f"   cho cat bin d={a['d']:+.5f} (t={a['t']:.2f})   bien bi loai d={b['d']:+.5f} (t={b['t']:.2f})")

1. Doi random_state cua XGBoost, giu nguyen bo fold va sieu tham so:
   random_state=42    M3 Gini CV = 0.72814
   random_state=1     M3 Gini CV = 0.72843
   random_state=7     M3 Gini CV = 0.72856
   random_state=2024  M3 Gini CV = 0.72745
   random_state=99    M3 Gini CV = 0.72806
   do lech chuan 0.00043   bien do 0.00110

2. Co dinh MOT bo sieu tham so cho ca ba model, chi con dau vao thay doi:
   [depth4 lr0.10] M2=0.72466 M3=0.72814 M4=0.73253   cho cat bin d=+0.00349 (t=6.18)   bien bi loai d=+0.00439 (t=8.99)
   [depth5 lr0.05] M2=0.72344 M3=0.72727 M4=0.73136   cho cat bin d=+0.00383 (t=5.52)   bien bi loai d=+0.00409 (t=10.31)


### Trước đó: "tương tác" có thật là tương tác không

Bước scorecard → M2 đổi **hai** thứ cùng lúc chứ không phải một, và tôi suýt bỏ qua. Scorecard có
đúng **một hệ số cho mỗi biến**, tức hình dạng của biến bị khoá ở giá trị WOE đơn biến. Cây trên
cùng các cột WOE đó có thể gán một giá trị riêng cho **từng bin**, tức thêm khoảng 51 tham số
hình dạng. Vậy +0,0099 có thể là tương tác, có thể là tự do hình dạng, hoặc cả hai.

Tách được bằng một phép kiểm rẻ: **cây `max_depth=1`**. Cây gốc đơn chỉ tách trên một biến mỗi
lần nên nó là model **cộng tính có hình dạng tự do**, không tương tác nào theo định nghĩa. Số
vòng để 1200 và đã kiểm hội tụ: 600 vòng cho 0,71589, 1200 cho 0,71562, 2000 cho 0,71569, ba
mức chênh nhau ít hơn nhiễu hạt giống (phép kiểm này chạy ngoài notebook, trên một máy).

In [11]:
# Tach "tu do hinh dang" khoi "tuong tac" bang cay depth=1 (cong tinh, hinh dang tu do)
P1 = dict(max_depth=1, learning_rate=0.10, n_estimators=1200, min_child_weight=50, subsample=0.8, colsample_bytree=0.8)
P4 = dict(max_depth=4, learning_rate=0.10, n_estimators=250,  min_child_weight=50, subsample=0.8, colsample_bytree=0.8)
g1 = tree_model.cv_gini(Xs['M2_woe'][trv], yv[trv], P1)
g4 = tree_model.cv_gini(Xs['M2_woe'][trv], yv[trv], P4)
r0_ = cv_check.paired(base_m, g1); r_ = cv_check.paired(g1, g4)
print(f"scorecard (logistic tren cot WOE)   {base_m.mean():.5f}")
print(f"XGB depth=1 tren DUNG cot WOE do    {g1.mean():.5f}   tu do hinh dang = {r0_['d']:+.5f}"
      f"  t={r0_['t']:5.2f}  KTC[{r0_['ktc_lo']:+.5f}; {r0_['ktc_hi']:+.5f}]")
print(f"XGB depth=4 tren DUNG cot WOE do    {g4.mean():.5f}   tuong tac       = {r_['d']:+.5f}"
      f"  t={r_['t']:5.2f}  KTC[{r_['ktc_lo']:+.5f}; {r_['ktc_hi']:+.5f}]")

scorecard (logistic tren cot WOE)   0.71507
XGB depth=1 tren DUNG cot WOE do    0.71559   tu do hinh dang = +0.00052  t= 1.18  KTC[-0.00070; +0.00174]
XGB depth=4 tren DUNG cot WOE do    0.72466   tuong tac       = +0.00907  t= 9.85  KTC[+0.00651; +0.01163]


Tự do hình dạng đáng **+0,0005** với `t = 1,18`, khoảng tin cậy `[−0,0007; +0,0017]`, tức **không phân biệt được với 0**. Tương tác đáng **+0,0091** với `t = 9,9`. Hai kênh chênh nhau 17 lần.

Kết quả này có lý do cơ học và đáng nhớ: **cột WOE đã là log-odds thực nghiệm của từng bin**, nên
model tuyến tính theo WOE vốn đã có sẵn hình dạng cộng tính tối ưu, sai khác đúng một hệ số co
giãn cho mỗi biến. Một cây `depth=1` học lại hình dạng đó từ đầu không có gì để thêm. Nói cách
khác, chính cách mã hoá WOE là thứ làm cho "tự do hình dạng" trở nên vô ích, và đó là một lập
luận bênh vực WOE mà tôi chưa có trước phép kiểm này.

Vậy +0,0099 ở bảng trên gần như **toàn bộ là tương tác**: +0,0091 tương tác cộng +0,0005 không đọc được.

Một ghi chú về việc sửa: bản đầu của phép kiểm này cho tự do hình dạng **−0,0008**, và tôi đã viết cả một đoạn giải thích vì sao nó âm ("cây học lại chỉ thêm nhiễu"). Con số âm đó là **artefact của early stopping**, không phải của dữ liệu: cây `depth=1` với `lr=0,10` cần hàng trăm vòng, mà early stopping cắt nó sớm. Bài học: một lời giải thích cơ học nghe hợp lý vẫn có thể đang giải thích cho một lỗi cài đặt.

### Cái gì ổn định và cái gì không

**Ngưỡng đọc được.** Đổi `random_state` mà giữ nguyên bộ fold và siêu tham số, Gini CV của M3 dao động với độ lệch chuẩn **0,00043**, biên độ **0,00110**. Tôi lấy **0,002** làm ngưỡng: con số này **không suy ra được** từ 0,00110, nó là biên độ làm tròn lên gần gấp đôi cho an toàn. Phép đo nhiễu này cũng chỉ đo nhiễu của **một lần fit** tại cấu hình đã thắng, không đo nhiễu của cả quy trình "chọn rồi báo cáo". Khi ngưỡng 0,002 và kiểm định `t` nói ngược nhau thì **không đọc con số đó**.

**Ổn định, đọc được:**

- **XGBoost hơn scorecard 0,0143 Gini trên cùng 9 biến** (t = 19,2), gấp mười mấy lần mức nhiễu.
- **Tương tác là kênh lớn nhất, +0,0091** (t = 9,9); tự do hình dạng không đọc được.
- Chỗ cắt bin và biến bị loại mỗi kênh đáng khoảng **+0,004**, và cả hai đều đọc được.

**Ít ổn định hơn:** *thứ tự* giữa hai kênh sau. Cố định một bộ siêu tham số cho cả ba model:

| cấu hình | chỗ cắt bin (M2→M3) | biến bị loại (M3→M4) |
|---|---|---|
| `max_depth=4`, `lr=0,10`, 250 vòng | +0,00349 (t = 6,18) | +0,00439 (t = 8,99) |
| `max_depth=5`, `lr=0,05`, 500 vòng | +0,00383 (t = 5,52) | +0,00409 (t = 10,31) |

Bốn con số nằm trong khoảng hẹp 0,0035 đến 0,0044 và cả bốn đều có `t` lớn, nên **độ lớn của hai kênh là kết luận đọc được**. Cái không đọc được chỉ là kênh nào lớn hơn, vì hai kênh gần bằng nhau và chênh lệch giữa chúng (0,0009 và 0,0003) nhỏ hơn mức tôi phân biệt được.

Câu đúng: **phần vượt của cây chủ yếu là tương tác (+0,0091), phần còn lại chia gần đều cho chỗ cắt bin và biến bị loại, mỗi kênh khoảng +0,004.**

Ở `results/scorecard.md` tôi viết "nếu chênh lệch chỗ cắt bin lớn hơn chênh lệch tương tác thì kết luận là chỗ cắt bin quan trọng hơn"; nó không lớn hơn ở bất kỳ cấu hình nào (0,004 so với 0,0091), nên kết luận đó vẫn đứng. Dự đoán B1 và B2 đều **trúng**, và tổng phần vượt nằm xa dưới ngưỡng 0,04 tôi đặt làm mốc nghi cài sai hoặc leakage.

### Chuyện tái lập, và một lỗi cài đặt phải sửa

Bản đầu của `tree_model.cv_gini` dùng `early_stopping_rounds=30` trên 20% cuối của fold-fit. Chạy trên Windows và trên Linux cho ra kết quả khác nhau **ở chữ số thứ ba**, và tệ hơn, **cấu hình thắng đổi theo nền tảng** (M3 thắng ở `depth=4` bên này, `depth=5` bên kia). Bản thân chênh lệch dấu phẩy động thì không lạ: `tree_method="hist"` cộng dồn gradient theo thứ tự phụ thuộc số luồng. Lạ là nó bị khuếch đại lên tới chữ số thứ ba. Chẩn đoán:

| | `depth=5, lr=0,05` |
|---|---|
| có early stopping | `best_iteration` mỗi fold = 31, 75, **8**, 98, 156; đổi seed → biên độ **0,00458** |
| số vòng cố định | đổi seed → biên độ **0,00036** |

Một fold dừng ở vòng **8** với `lr=0,05` là một model chưa học gì. Eval set chỉ khoảng 16.800 dòng với ~1.120 ca dương nên AUC trên nó rất nhiễu, và 30 vòng là quá ngắn để phân biệt một đỉnh giả với một đỉnh thật. Early stopping ở đây **không chống overfit mà biến nhiễu của eval set thành nhiễu của kết quả, gấp 13 lần.**

Bỏ nó đi được thêm hai thứ. Cây giờ fit trên **trọn 80%** mỗi fold, đúng bằng scorecard, nên mất một thiên lệch mà trước đó tôi phải khai mà không tách được. Và bảng test ở cuối notebook dùng đúng số vòng của bảng CV, nên hai bảng đọc chéo nhau được. Cái giá: số vòng thành một siêu tham số nữa, được khai trước theo quy tắc `learning_rate × n_estimators ≈ 25` chứ không dò tìm.

Đây là lỗi tốn của tôi nhiều thời gian nhất trong khối này, và nó đã sinh ra ít nhất ba kết luận sai mà tôi đã viết ra rồi phải rút: dấu của "tự do hình dạng", quy luật "cây nông thì lợi rơi vào chỗ cắt bin", và mức chênh lệch giữa hai nền tảng.

---
## 4. Câu hỏi riêng: `open_credit_lines`

Khối 3 loại biến này khỏi scorecard vì đóng góp biên của nó là **+0,00004**, tức bỏ đi thì Gini còn nhích lên. M4 thêm nó cùng `debt_ratio` thô vào cây, và phần tăng cần tách xem của biến nào.

In [12]:
P3 = chon['M3_goc'][0]
b3 = chon['M3_goc'][1]
print(f'M3 (9 bien goc + co){"":22s} {b3.mean():.5f}')
for extra in [['open_credit_lines'], ['debt_ratio'], ['open_credit_lines', 'debt_ratio']]:
    Z = pd.concat([Xs['M3_goc'], ap_[extra]], axis=1)
    gg = tree_model.cv_gini(Z[trv], yv[trv], P3); r = cv_check.paired(b3, gg)
    print(f"M3 + {'+'.join(extra):34s} {gg.mean():.5f}   d = {r['d']:+.5f}  t = {r['t']:5.2f}  "
          f"KTC[{r['ktc_lo']:+.5f}; {r['ktc_hi']:+.5f}]  5 fold cung dau = {r['cung_dau']}")

M3 (9 bien goc + co)                       0.72935
M3 + open_credit_lines                  0.73237   d = +0.00303  t =  7.41  KTC[+0.00189; +0.00416]  5 fold cung dau = True
M3 + debt_ratio                         0.72942   d = +0.00007  t =  0.20  KTC[-0.00088; +0.00102]  5 fold cung dau = False
M3 + open_credit_lines+debt_ratio       0.73280   d = +0.00346  t =  8.26  KTC[+0.00230; +0.00462]  5 fold cung dau = True


Toàn bộ phần tăng của M4 là của `open_credit_lines` (+0,0030, t = 7,4, năm fold cùng dấu): `debt_ratio` thô không phân biệt được với 0 (+0,0001, t = 0,2).

Nhưng **kết luận đầu tiên tôi rút ra từ đây là sai**, và cách nó sai đáng ghi lại.

Tôi viết rằng "toàn bộ giá trị của biến này nằm ở tương tác", lấy lý do là nó đáng +0,00004 với model cộng tính ở khối 3 và +0,0030 với cây ở đây. Nhưng hai con số đó khác nhau ở **ba** thứ cùng lúc chứ không phải một: lớp hàm (cộng tính so với cây), dạng biến (cột WOE của khối 2 so với biến gốc), và mốc so. Chỉ thứ nhất là tương tác. Suy từ chênh lệch ba-thứ-đổi ra kết luận về một thứ là đúng cái lỗi tôi đã mắc ở khối 3 với chữ U.

Phép kiểm tách được: cho `open_credit_lines` vào một model **cộng tính** (cây `depth=1`) theo hai cách.

In [13]:
# open_credit_lines: gia tri cua no la tuong tac, hay la hieu ung chinh bi ma hoa WOE giet?
fw2 = pd.read_sql('SELECT id, woe_open_credit_lines FROM features_woe',
                  sqlite3.connect(scorecard._ro_uri(config.DB_PATH), uri=True)
                  ).sort_values('id').reset_index(drop=True)
b1 = tree_model.cv_gini(Xs['M2_woe'][trv], yv[trv], P1)
print(f"9 cot WOE don dieu (dau vao cua scorecard)          {b1.mean():.5f}")
for nhan, Zx in [('  + cot WOE cua no (ma hoa khoi 2)', pd.concat([Xs['M2_woe'], fw2[['woe_open_credit_lines']]], axis=1)),
                 ('  + bien GOC (cay tu chon hinh dang)', pd.concat([Xs['M2_woe'], ap_[['open_credit_lines']]], axis=1))]:
    gx = tree_model.cv_gini(Zx[trv], yv[trv], P1); rx = cv_check.paired(b1, gx)
    print(f"{nhan:52s} {gx.mean():.5f}   d={rx['d']:+.5f}  t={rx['t']:5.2f}  "
          f"KTC[{rx['ktc_lo']:+.5f}; {rx['ktc_hi']:+.5f}]")

9 cot WOE don dieu (dau vao cua scorecard)          0.71559
  + cot WOE cua no (ma hoa khoi 2)                   0.71792   d=+0.00233  t= 4.37  KTC[+0.00085; +0.00381]
  + bien GOC (cay tu chon hinh dang)                 0.71845   d=+0.00286  t= 3.48  KTC[+0.00058; +0.00514]


Hai con số **gần nhau**: cho biến vào ở dạng thô hay ở dạng cột WOE của khối 2 thì model cộng tính đều rút ra được khoảng 0,0023 đến 0,0029 Gini, và chênh lệch giữa hai cách nằm trong nhiễu.

Vậy bin của khối 2 **không** phải chỗ tín hiệu bị mất, và đây là lần thứ hai tôi phải sửa kết luận về biến này. Chỗ mất nằm ở đâu thì phải hỏi thẳng bằng một phép chỉ đổi **một** thứ: cùng bộ bin của khối 2, cùng logistic, chỉ đổi số tham số dành cho biến đó.

In [14]:
# Cung bo bin khoi 2, cung logistic, chi doi SO THAM SO cho open_credit_lines.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
con_ = sqlite3.connect(scorecard._ro_uri(config.DB_PATH), uri=True)
rb_  = pd.read_sql("SELECT id, bin FROM row_bins WHERE variable='open_credit_lines'", con_).sort_values('id').reset_index(drop=True)
fw2_ = pd.read_sql('SELECT id, woe_open_credit_lines w FROM features_woe', con_).sort_values('id').reset_index(drop=True)
con_.close()
Wm = scorecard.woe_matrix(bins_m, woe_m)
assert (Wm.index.values == rb_.id.values).all(), 'thu tu dong khong khop'
m_tr = (bins_m.split == 'train').values
Wtr, ytr = Wm[m_tr].reset_index(drop=True), (1 - bins_m.target).values[m_tr]
D = pd.get_dummies(rb_.bin.values[m_tr], prefix='b', drop_first=True).astype(float)
folds_ = cv_check.make_folds(len(Wtr), 5, None)

def cv_logit(X):
    X = np.asarray(X, dtype=float); g = []
    for f in range(5):
        va = folds_[f]; fit = np.concatenate([folds_[i] for i in range(5) if i != f])
        lr_ = LogisticRegression(C=1e12, max_iter=2000).fit(X[fit], ytr[fit])
        g.append(2 * roc_auc_score(ytr[va], lr_.predict_proba(X[va])[:, 1]) - 1)
    return np.array(g)

bl = cv_logit(Wtr)
print(f'9 cot WOE don dieu (dau vao scorecard)              {bl.mean():.5f}')
for nhan, X in [(f'  + 1 cot WOE cua open_credit_lines (1 he so)', np.column_stack([Wtr.values, fw2_.w.values[m_tr]])),
                (f'  + {D.shape[1]} cot dummy theo bin cua no (cung bin do)', np.column_stack([Wtr.values, D.values]))]:
    g = cv_logit(X); rr = cv_check.paired(bl, g)
    print(f"{nhan:52s} {g.mean():.5f}   d={rr['d']:+.5f}  t={rr['t']:5.2f}  KTC[{rr['ktc_lo']:+.5f}; {rr['ktc_hi']:+.5f}]")

9 cot WOE don dieu (dau vao scorecard)              0.71567
  + 1 cot WOE cua open_credit_lines (1 he so)        0.71591   d=+0.00024  t= 2.36  KTC[-0.00004; +0.00052]
  + 9 cot dummy theo bin cua no (cung bin do)        0.71774   d=+0.00207  t= 3.88  KTC[+0.00059; +0.00355]


Cùng một cách chia bin, cùng một model, khác đúng **một** thứ là số tham số: một hệ số cho +0,0002 (khoảng tin cậy chứa 0), chín cột dummy cho **+0,0021** (t = 3,9). Gấp gần chín lần.

Con số +0,0021 này khớp với +0,0023 mà cây `depth=1` rút ra được từ chính cột WOE đó, tức hai đường hoàn toàn khác nhau cho cùng một câu trả lời.

**Kết luận, sau khi đã sai hai lần:**

| cách đưa `open_credit_lines` vào | đóng góp |
|---|---|
| biến gốc, model cộng tính tự chọn hình dạng | +0,0029 |
| cột WOE khối 2, model cộng tính (`depth=1`) | +0,0023 |
| cột WOE khối 2, logistic với **9 cột dummy** | +0,0021 |
| cột WOE khối 2, logistic với **1 hệ số** (scorecard khối 3) | **+0,00004** |

Ba dòng đầu nói cùng một chuyện; dòng cuối rơi xuống gần 0. Thủ phạm **không phải cách chia bin và không phải thiếu tương tác**, mà là **ràng buộc một hệ số cho cả cột WOE**, tức chính dạng hàm của scorecard cổ điển. Cột WOE của biến này có hình chữ U, mà khối 3 đã chứng minh nhánh trái chỉ là bóng của `revolving_util`; nhân cả cột với một hằng số thì phần còn có ích không tách ra được.

Quyết định loại nó ở khối 3 vì thế **đúng với dạng hàm của scorecard**, và cách sửa ở khối 5 là chia bin lại sao cho cột WOE đơn điệu (khi đó một hệ số là đủ) hoặc cho biến này nhiều hơn một tham số. Điểm mù của quy trình "chia bin một lần rồi chọn biến" vẫn đứng, chỉ là phát biểu lại cho đúng: **một biến mà scorecard không dùng được chưa chắc là một biến vô dụng; có thể chỉ là scorecard không đủ tham số cho nó.**

Điều vẫn đứng từ đầu: hai kết luận ngược nhau về cùng một biến không mâu thuẫn, vì chúng nói về hai lớp hàm khác nhau. Ranh giới không nằm ở "có tương tác hay không" như tôi tưởng lần đầu, cũng không ở "bin tốt hay xấu" như tôi tưởng lần hai, mà ở **số tham số mà model cho phép biến đó có**.

---
## 5. `scale_pos_weight` và cái giá của nó

Giả thuyết số 1 ghi ở `notes_credit_scoring.md` §10 từ khối 0: bật và tắt `scale_pos_weight` chênh nhau dưới 0,01 Gini. Kiểm luôn cả tác động lên calibration, vì đó mới là chỗ nó gây hại.

In [15]:
import xgboost as xgb
Z4 = pd.concat([Xs['M3_goc'], ap_[['open_credit_lines', 'debt_ratio']]], axis=1)
P4 = chon['M4_all'][0]
w = (1 - yv[trv]).sum() / yv[trv].sum()          # n_bad / n_good ; y = 1 la GOOD
print(f'ty le bad/good tren train = {w:.5f}   -ln(w) = {-np.log(w):.4f}\n')
b4 = chon['M4_all'][1]
gw = tree_model.cv_gini(Z4[trv], yv[trv], dict(P4, scale_pos_weight=w))
r  = cv_check.paired(b4, gw)
print(f'tat : Gini CV = {b4.mean():.5f}')
print(f'bat : Gini CV = {gw.mean():.5f}   d = {r["d"]:+.5f}  t = {r["t"]:5.2f}  '
      f'KTC[{r["ktc_lo"]:+.5f}; {r["ktc_hi"]:+.5f}]\n')
for tag, prm in [('tat', P4), ('bat', dict(P4, scale_pos_weight=w))]:
    m = xgb.XGBClassifier(tree_method='hist', eval_metric='auc',
                          random_state=config.SEED, **prm).fit(Z4[trv], yv[trv])
    pdb = 1 - m.predict_proba(Z4[trv])[:, 1]
    print(f'  {tag}: PD du bao TB = {pdb.mean()*100:6.3f}%   bad rate thuc = {(1-yv[trv]).mean()*100:.3f}%'
          f'   ty le = {pdb.mean()/(1-yv[trv]).mean():.2f}x')

ty le bad/good tren train = 0.07163   -ln(w) = 2.6363

tat : Gini CV = 0.73280
bat : Gini CV = 0.73200   d = -0.00080  t = -1.44  KTC[-0.00234; +0.00074]

  tat: PD du bao TB =  6.685%   bad rate thuc = 6.684%   ty le = 1.00x
  bat: PD du bao TB = 31.981%   bad rate thuc = 6.684%   ty le = 4.78x


Giả thuyết đúng: chênh lệch Gini là **−0,00080** với `t = −1,44` và khoảng tin cậy `[−0,0023; +0,0007]`, dưới ngưỡng 0,01 rất xa.

Nhưng calibration vỡ hẳn: PD dự báo trung bình nhảy từ 6,685% lên **31,981%**, tức **4,78 lần** bad rate thực. Hệ số cân bằng lớp mua độ nhạy bằng cách dịch intercept đi một lượng liên quan tới `−ln(w) = 2,64`, và cái giá là mọi con số PD mất nghĩa. (Mức dịch quan sát được nhỏ hơn một phép dịch logit thuần, vì `base_score` và cấu trúc cây hấp thụ một phần; tôi không khẳng định con số chính xác vì chưa tách được hai phần đó.)

Kết luận triển khai: với bài toán này **không bật `scale_pos_weight`**. Nó không mua được thứ tự xếp hạng, mà thứ nó phá là đúng cái scorecard tồn tại để cung cấp.

---
## 6. SHAP: tương tác nằm ở đâu

Giá trị SHAP interaction lấy thẳng từ XGBoost (`pred_interactions=True`) chứ không qua gói `shap`, vì `shap` 0.49 chưa đọc được `base_score` dạng mảng của XGBoost 3.x. Tính trên mẫu 4.000 dòng train, đây là thông lệ vì chi phí của interaction values là bậc hai theo số biến.

In [16]:
mX = xgb.XGBClassifier(tree_method='hist', eval_metric='auc',
                       random_state=config.SEED, **P4).fit(Z4[trv], yv[trv])
rng = np.random.default_rng(config.SEED)
idx = rng.choice(np.where(trv)[0], 4000, replace=False)
SI = mX.get_booster().predict(xgb.DMatrix(Z4.iloc[idx]), pred_interactions=True)
cols = list(Z4.columns); A = np.abs(SI).mean(0)[:len(cols), :len(cols)]
print('Dong gop chinh (|SHAP| trung binh, duong cheo):')
for k, v in pd.Series(np.diag(A), index=cols).sort_values(ascending=False).head(6).items():
    print(f'   {k:26s} {v:.4f}')
off = A.copy(); np.fill_diagonal(off, 0)
pairs = [(cols[i], cols[j], off[i, j]*2) for i in range(len(cols)) for j in range(i+1, len(cols))]
print('\nMuoi cap tuong tac manh nhat:')
for a, b, v in sorted(pairs, key=lambda x: -x[2])[:10]:
    print(f'   {a:24s} x {b:22s} {v:.5f}')
tmain, tint = np.diag(A).sum(), off.sum()
print(f'\ntong |main| = {tmain:.4f}   tong |tuong tac| = {tint:.4f}   '
      f'tuong tac chiem {tint/(tmain+tint)*100:.1f}%')

Dong gop chinh (|SHAP| trung binh, duong cheo):
   revolving_util             0.8181
   late_30_59                 0.3456
   late_90                    0.2715
   age                        0.2084
   late_60_89                 0.1660
   open_credit_lines          0.1409

Muoi cap tuong tac manh nhat:
   revolving_util           x age                    0.04811
   revolving_util           x late_30_59             0.04428
   age                      x late_30_59             0.03974
   age                      x monthly_income         0.03876
   revolving_util           x debt_ratio_valid       0.03864
   revolving_util           x late_90                0.03646
   age                      x open_credit_lines      0.03246
   late_30_59               x late_60_89             0.03161
   revolving_util           x open_credit_lines      0.02906
   debt_ratio_valid         x real_estate_loans      0.02765

tong |main| = 2.3250   tong |tuong tac| = 0.9346   tuong tac chiem 28.7%


Dự đoán B4 **không kết luận được**, và lý do đáng ghi hơn cả bản thân dự đoán.

Tôi đoán cặp mạnh nhất là `revolving_util` với một trong ba biến `late_*`. Ba lần chạy khác nhau (Windows, Linux, và một lần nữa sau khi sửa early stopping) cho **ba cặp đứng đầu khác nhau**: `revolving_util × debt_ratio_valid`, `revolving_util × late_30_59`, rồi `revolving_util × age`. Ở lần chạy được lưu, bốn cặp đầu bảng nằm trong khoảng hẹp 0,039 đến 0,048, tức cách nhau ít hơn mức tôi phân biệt được.

**Một dự đoán mà phán quyết đổi theo hạt giống ngẫu nhiên thì không phải một dự đoán đặt đúng.** Cách đặt đúng lẽ ra phải là "cặp `revolving_util × late_*` nằm trong ba cặp mạnh nhất", một mệnh đề đủ thô để không bị nhiễu lật, và mệnh đề đó **đúng ở cả ba lần chạy**. Đây là lỗi thiết kế dự đoán chứ không phải lỗi đo: tôi đã đặt một câu hỏi mà dữ liệu ở quy mô này không trả lời được.

Hai điều ổn định hơn và đáng chú ý hơn.

`open_credit_lines` là **đóng góp chính mạnh thứ sáu** trong bảng SHAP ở cả ba lần chạy, và có mặt ở hai đến ba trong mười cặp tương tác mạnh nhất, dù đóng góp của nó trong scorecard bằng không. SHAP và CV nói cùng một chuyện qua hai đường khác nhau.

Tương tác chiếm **28,7%** tổng |SHAP| nhưng kênh tương tác chỉ quy ra +0,0091 Gini. Hai con số không mâu thuẫn, và phải nói kèm hai điều. Thứ nhất chúng là **của hai model khác nhau**: 28,7% đo trên M4 (biến thô, thêm hai biến), còn +0,0091 đo trên M2 (cột WOE). Thứ hai, ngay trong cùng một model thì phần lớn tương tác là tinh chỉnh quanh một hiệu ứng chính rất mạnh chứ không tạo ra thứ tự xếp hạng mới. Đây là lý do không nên đọc tỉ lệ SHAP như thể nó là tỉ lệ sức mạnh dự báo.

---
## 7. Tập test, dùng đúng một lần

Mọi quyết định ở trên đã ra đời trong train. Mục này chỉ để đối chiếu với dự đoán ghi trước và để báo cáo.

In [17]:
tev = split == 'test'
def dm(nm, Z, prm):
    m = xgb.XGBClassifier(tree_method='hist', eval_metric='auc',
                          random_state=config.SEED, **prm).fit(Z[trv], yv[trv])
    return nm, m.predict_proba(Z[trv])[:, 1], m.predict_proba(Z[tev])[:, 1]
sm = S['mono']
res = [('scorecard (logistic, WOE don dieu)',
        sm['lr'].predict_proba(sm['W'][sm['tr']])[:, 1], sm['lr'].predict_proba(sm['W'][sm['te']])[:, 1])]
res += [dm('M2 XGB tren cot WOE', Xs['M2_woe'], chon['M2_woe'][0]),
        dm('M3 XGB tren bien goc', Xs['M3_goc'], chon['M3_goc'][0]),
        dm('M4 XGB tren tat ca',   Z4,           P4)]
rows = []
for nm, ptr, pte in res:
    rows.append({'model': nm,
                 'train Gini': scorecard.gini(yv[trv], ptr), 'test Gini': scorecard.gini(yv[tev], pte),
                 'test KS': scorecard.ks(yv[tev], pte),
                 'chenh train-test': scorecard.gini(yv[trv], ptr) - scorecard.gini(yv[tev], pte),
                 'PD TB test %': (1 - pte).mean() * 100})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print(f"bad rate thuc tren test = {(1-yv[tev]).mean()*100:.3f}%")

# Khoang tin cay cua chinh Gini tren test, de biet cai bang tren doc duoc den dau.
# Hanley & McNeil (1982) cho SE cua AUC; Gini = 2*AUC - 1 nen SE(Gini) = 2*SE(AUC).
A = (scorecard.gini(yv[tev], res[0][2]) + 1) / 2
n1, n2 = int((1 - yv[tev]).sum()), int(yv[tev].sum())
Q1, Q2 = A / (2 - A), 2 * A * A / (1 + A)
seA = np.sqrt((A * (1 - A) + (n1 - 1) * (Q1 - A * A) + (n2 - 1) * (Q2 - A * A)) / (n1 * n2))
rg = np.random.default_rng(config.SEED); yt = yv[tev]; pt = res[0][2]; bs = []
for _ in range(300):
    j = rg.integers(0, len(yt), len(yt)); bs.append(scorecard.gini(yt[j], pt[j]))
print(f"\ntest: {n1} ca duong / {n2} ca am")
print(f"KTC95 nua rong cua Gini: Hanley-McNeil +-{1.96*2*seA:.4f}   bootstrap 300 lan +-{1.96*np.std(bs, ddof=1):.4f}")

                             model  train Gini  test Gini  test KS  chenh train-test  PD TB test %
scorecard (logistic, WOE don dieu)      0.7162     0.7006   0.5440            0.0155        6.5985
               M2 XGB tren cot WOE      0.7332     0.7139   0.5732            0.0193        6.6073
              M3 XGB tren bien goc      0.7455     0.7137   0.5692            0.0317        6.5639
                M4 XGB tren tat ca      0.7500     0.7208   0.5752            0.0292        6.5753
bad rate thuc tren test = 6.684%

test: 1504 ca duong / 20996 ca am
KTC95 nua rong cua Gini: Hanley-McNeil +-0.0247   bootstrap 300 lan +-0.0214


Ba điều đọc được, và một điều không.

**Đọc được.** Scorecard có khoảng cách train trừ test hẹp nhất (0,0155); cả ba model cây rộng hơn (0,019 đến 0,032), đúng chữ ký của model nhiều tham số hơn. PD dự báo trung bình của cả bốn model đều quanh 6,55 đến 6,60% so với bad rate thực 6,684%, tức không model nào lệch trung bình đáng kể khi `scale_pos_weight` tắt. Và bảng này giờ **đọc chéo được với bảng CV**, vì sau khi bỏ early stopping thì cả hai dùng đúng cùng một cấu hình và cùng số vòng.

**Không đọc được: thứ tự giữa ba model cây.** Trên CV thì M3 (0,72935) hơn M2 (0,72492) rất rõ (t = 10,0); trên test thì hai model gần như bằng nhau và M2 nhỉnh hơn (0,7139 so với 0,7137). Chênh lệch đó nằm sâu trong khoảng tin cậy của chính Gini trên test, nên đây không phải mâu thuẫn mà là hai lần rút thăm từ cùng một phân phối.

Con số nên tin vẫn là CV chứ không phải test: test chỉ có 1.504 ca dương nên khoảng tin cậy 95% của Gini là **±0,025** theo Hanley–McNeil (bootstrap 300 lần cho ±0,021), tức rộng hơn toàn bộ chênh lệch 0,020 giữa scorecard và model cây tốt nhất. Con số ±0,028 dùng ở khối 3 là cho tập OOT với cùng số ca dương nhưng tính bảo thủ hơn; hai chỗ nên thống nhất ở khối 5.

---
## Một hệ quả của đơn điệu chưa nói ở mục 1

Ép đơn điệu được bán bằng lập luận giải thích được. Với `revolving_util` thì đúng. Với `real_estate_loans` thì nó **tạo ra** một vấn đề giải thích mới.

In [18]:
# Bad rate tho va diem truoc/sau khi ep don dieu, cho real_estate_loans
V0 = 'real_estate_loans'
ap1 = pd.read_sql('SELECT id, split, target FROM applications',
                  sqlite3.connect(scorecard._ro_uri(config.DB_PATH), uri=True))
t1 = ap1[ap1.split == 'train'].set_index('id')
d_goc  = dict(zip(S['goc']['pts'].query('variable == @V0').bin,  S['goc']['pts'].query('variable == @V0').diem))
d_mono = dict(zip(S['mono']['pts'].query('variable == @V0').bin, S['mono']['pts'].query('variable == @V0').diem))
tab = pd.DataFrame({'bin_goc': bins_g.loc[t1.index, V0], 'bin_mono': bins_m.loc[t1.index, V0],
                    'target': t1.target})
g = tab.groupby(['bin_goc', 'bin_mono']).agg(n=('target', 'size'), bad_rate=('target', 'mean')).reset_index()
g['bad_rate'] = (g.bad_rate * 100).round(2)
g['diem_goc'] = g.bin_goc.map(d_goc)
g['diem_mono'] = g.bin_mono.map(d_mono)
print(g.to_string(index=False))
print(f"\nHai nhom co bad rate tho gan bang nhau (bin 0 va bin 3+) cach nhau "
      f"{g.loc[g.bin_goc == '0', 'diem_mono'].iloc[0] - g.loc[g.bin_goc == '3+', 'diem_mono'].iloc[0]:.0f} diem "
      f"o bang don dieu, so voi {g.loc[g.bin_goc == '0', 'diem_goc'].iloc[0] - g.loc[g.bin_goc == '3+', 'diem_goc'].iloc[0]:.0f} diem o bang cu.")

bin_goc bin_mono     n  bad_rate  diem_goc  diem_mono
      0    0+1+2 39283      8.33        59         64
      1    0+1+2 36645      5.23        67         64
      2    0+1+2 22087      5.60        65         64
     3+       3+  6984      8.46        58         49

Hai nhom co bad rate tho gan bang nhau (bin 0 va bin 3+) cach nhau 15 diem o bang don dieu, so voi 1 diem o bang cu.


Bảng điểm cũ chấm nhóm 8,33% và nhóm 8,46% gần bằng nhau (59 và 58), đúng như bad rate thô của họ. Bảng điểm "giải thích được" mới tách hai nhóm gần như cùng rủi ro thô ra **15 điểm**, và cho nhóm 0 khoản điểm tối đa của biến.

Về thống kê thì bảo vệ được: khối 3 đã đo rằng rủi ro thừa của nhóm 0 khoản đã nằm trong các biến khác, nên sau khi đã có 8 biến kia thì nhóm đó **không** rủi ro hơn thật. Nhưng đó là một câu trả lời **đa biến**, và nó không nói được với một khách hàng bị từ chối, trong khi giải thích được cho khách hàng chính là lý do cả bước đơn điệu hoá tồn tại.

Ghi lại làm giới hạn đã biết: cái giá thật của đơn điệu hoá ở biến này không nằm ở Gini.

---
## Giới hạn đã biết

- Siêu tham số của cây chọn bằng chính bộ CV dùng để báo cáo, nên con số CV của cây lạc quan. Thiên lệch này nghiêng về phía cây, tức ngược với kết luận "cây chỉ hơn 0,0143".
- Cấu hình thắng hơn cấu hình nhì 0,00026 ở M2, 0,00121 ở M3, 0,00027 ở M4, tức cùng bậc với mức nhiễu 0,00110. Cấu hình thắng vì thế gần như tuỳ ý; điều cứu phép so sánh là cả ba model cùng thắng ở một cấu hình và các cấu hình đứng đầu cho gần cùng một con số, nên chọn cái nào cũng không đổi kết luận.
- Số vòng boosting là một siêu tham số được khai trước theo quy tắc `lr × n_estimators ≈ 25` chứ không dò tìm, nên nó không tối ưu cho từng cấu hình. Đây là cái giá phải trả để bỏ early stopping, và tôi chọn trả nó.
- Bin của scorecard (điểm cắt `NTILE` và nhóm PAVA) học trên toàn bộ train rồi dùng lại trong mọi fold, trong khi cây học mọi thứ trong fold. Riêng nhóm PAVA được sinh ra **từ WOE, tức từ nhãn**, nên lý lẽ đã dùng ở khối 3 để tha cho điểm cắt `NTILE` (phân vị của biến, không nhìn nhãn) không mở rộng sang đây được. Thiên lệch này có lợi cho scorecard.
- Điểm cắt bin vẫn không được tính lại trong từng fold, chỉ WOE mới được.
- Mã lý do vẫn tập trung 74,3% vào một biến duy nhất.
- Tập `oot` vẫn chưa được đọc nhãn ở bất kỳ phép đo nào.

---
## Xong bước này

| | |
|---|---|
| Chia bin | 70 → 60 bin, đơn điệu theo chiều nghiệp vụ khai báo |
| Cái giá của đơn điệu | 0,0019 Gini ở cận trên KTC 95%, tức không đo được |
| Scorecard đơn điệu | test Gini 0,7006, KS 0,5440, bảng điểm 60 dòng, 376–636 |
| Mã lý do tập trung | 84,7% → 74,3% |
| So sánh chính (cùng 9 biến) | **XGBoost hơn scorecard 0,0143 Gini** (t = 19,2) |
| Phân rã phần hơn | tự do hình dạng +0,0005 (không đọc được), **tương tác +0,0091**, chỗ cắt bin +0,004, biến bị loại +0,004 |
| Câu hỏi riêng | `open_credit_lines` mang hiệu ứng **chính** đáng +0,002 đến +0,003; thứ giết nó là **ràng buộc một hệ số**, không phải cách chia bin |
| `scale_pos_weight` | không đổi Gini đáng kể nhưng PD sai 4,8 lần |
| Mức nhiễu của cây | đổi `random_state` làm Gini CV dao động biên độ 0,0011 |

Chín dự đoán ghi trước ở `notes/du_doan_khoi4.md`: **năm trúng, một trúng một phần** (A2), **hai trượt** (A4 và A5, cả hai sát mép và về phía kết quả tốt hơn tôi tưởng), **một không kết luận được** (B4, vì phán quyết đổi theo hạt giống ngẫu nhiên).

Ngoài ra khối này có **ba kết luận tôi viết ra rồi phải rút lại**, và cả ba đều đáng ghi hơn bản thân kết luận đúng: "giá trị của `open_credit_lines` nằm ở tương tác" (sai, nó là hiệu ứng chính), "mã hoá WOE của khối 2 giết 60% tín hiệu của nó" (sai, thủ phạm là ràng buộc một hệ số), và "tự do hình dạng đáng −0,0008" (sai, đó là artefact của early stopping). Hai cái đầu cùng một kiểu lỗi: suy từ một chênh lệch nhiều-thứ-đổi ra kết luận về một thứ. Cái thứ ba là một lỗi cài đặt mà tôi đã kịp viết cả một lời giải thích cơ học nghe hợp lý cho nó.

Bước tiếp theo là khối 5: calibration (Platt và isotonic), PSI và CSI, tập OOT dùng lần đầu, và một OOT dịch nhân tạo để PSI có cái để bắt. Ba giả thuyết còn lại ở `notes_credit_scoring.md` §10 sẽ được kiểm ở đó, cùng với việc chia bin lại cho `open_credit_lines`.